# Notebook 10: Judge + Analyze

Consumes the 9 CSVs produced by notebooks 1–9. Runs the Gemini 3 Flash
judge over every (clip, model) pair, then builds the master JSON,
comparison table, plots, and final report.

**Flow:**

1. Clone repo, install minimal deps.
2. Rebuild the manifest deterministically (same `seed=42` as the model notebooks).
3. **Upload your 9 prediction CSVs as a Kaggle dataset** (instructions below).
4. Aggregate uploaded CSVs into `./predictions/`.
5. Set `OPENROUTER_API_KEY` in Kaggle Secrets.
6. Judge → master JSON → comparison table → plots → report → bundle zip.


In [ ]:
# === Clone the benchmark repo ===========================================
# Kaggle paths: /kaggle/working is the writable home. We clone the project
# repo there and cd into it so the relative paths used by src/ resolve.

!apt-get -qq install -y libsndfile1

GITHUB_REPO_URL = "https://github.com/notAvailable73/stt-model-comparison.git"
REPO_DIR = "/kaggle/working/banglish-stt-benchmark"

import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# === Install minimal deps (no torch, no transformers) ===================
!pip install -q openai==1.55.0 pandas==2.2.3 pyyaml==6.0.2 \
    matplotlib==3.9.2 seaborn==0.13.2 librosa==0.10.2.post1 \
    soundfile==0.12.1 tqdm==4.67.1 requests==2.32.3


In [ ]:
# === Build the 50-clip manifest =========================================
# Downloads OpenSLR-104 if not already cached, extracts, parses transcripts,
# computes length + code-switch density, and stratified-samples 50 clips.
#
# CRITICAL: every notebook in this benchmark calls this with seed=42 so
# they all see the EXACT SAME 50 clips. Do not change the seed.

from src.utils import setup_logging, set_seeds
from src.data_prep import build_manifest

setup_logging("logs")
set_seeds(42)

manifest_path = build_manifest(
    raw_dir="data/raw",
    manifest_path="data/manifest.csv",
    n_samples=50,
    seed=42,
)
print("manifest:", manifest_path)


## UPLOAD YOUR PREDICTIONS

After running notebooks 1–9, you have 9 CSVs saved locally. They are named
exactly:

```
openai__whisper-large-v3.csv
openai__whisper-large-v3-turbo.csv
KhushiDS__whisper-large-v3-Bengali.csv
ai4bharat__indicconformer_stt_bn_hybrid_ctc_rnnt_large.csv
hishab__titu_stt_bn_fastconformer.csv
hishab__titu_stt_bn_conformer_large.csv
facebook__omniASR-CTC-300M.csv
bangla-speech-processing__BanglaASR.csv
google__chirp-3.csv
```

**Steps on this Kaggle notebook:**

1. Right sidebar → **+ Add Data** → **Upload** (the tab to the right of "Search Datasets").
2. Drag in all 9 CSVs. Give the dataset a name like `banglish-stt-predictions`.
3. After upload finishes, the files appear at `/kaggle/input/<dataset-slug>/`.
4. Set `UPLOADED_DIR` in the next cell to that path.

The next cell will validate that all 9 expected slugs are present and copy them
into `./predictions/`. Any missing CSV is reported and the corresponding model
is marked `FAILED` in the final JSON — the judge / analysis still runs over the
models that ARE present.


In [ ]:
# === Aggregate uploaded CSVs into ./predictions/ ========================

from pathlib import Path
import shutil
from src.utils import read_yaml, slugify_model_id

UPLOADED_DIR = "/kaggle/input/banglish-stt-predictions"   # <- change to match your upload

local_dir = Path("predictions")
local_dir.mkdir(parents=True, exist_ok=True)

expected_slugs = {
    slugify_model_id(spec["id"]): spec["id"]
    for spec in read_yaml("config/models.yaml")["models"]
}

found, missing = [], []
upload_path = Path(UPLOADED_DIR)
for slug, model_id in expected_slugs.items():
    src = upload_path / f"{slug}.csv"
    dst = local_dir / f"{slug}.csv"
    if src.exists():
        shutil.copyfile(src, dst)
        found.append(model_id)
    else:
        missing.append(model_id)

print(f"Found {len(found)}/{len(expected_slugs)} prediction CSVs.")
if missing:
    print("Missing (will be marked FAILED):")
    for m in missing:
        print(" -", m)

# `failed_models` is passed to the analysis cells below.
failed_models = [
    {"model_id": m, "status": "FAILED", "error": "no predictions CSV uploaded"}
    for m in missing
]


In [ ]:
# === OPENROUTER_API_KEY ================================================
# Recommended: Add-ons → Secrets → add OPENROUTER_API_KEY (never logged).
# Fallback: getpass prompt.

import os
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENROUTER_API_KEY"] = (
            UserSecretsClient().get_secret("OPENROUTER_API_KEY")
        )
        print("Loaded OPENROUTER_API_KEY from Kaggle Secrets.")
    except Exception:
        from getpass import getpass
        os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")
print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))


In [ ]:
# === Judging ============================================================
# One Gemini 3 Flash call per (clip, model). Resumable: rows already in
# judgments.csv are skipped, so reruns continue from the last judged pair.

from src.judge import judge_predictions

judge_predictions(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    prompts_yaml="config/prompts.yaml",
)


In [ ]:
# === Master JSON ========================================================
from src.analyze import build_master_json

master_json = build_master_json(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    output_json="results/benchmark_results.json",
)
print("Master JSON:", master_json)


In [ ]:
# === Comparison table ===================================================
from src.analyze import build_comparison_table
from IPython.display import Image, display
import pandas as pd

comp_csv, comp_png = build_comparison_table(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    out_csv="results/comparison_table.csv",
    out_png="results/comparison_table.png",
)
display(pd.read_csv(comp_csv))
display(Image(str(comp_png)))


In [ ]:
# === Analytical plots ===================================================
from src.analyze import make_all_plots
from IPython.display import Image, display

plot_paths = make_all_plots(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    plots_dir="results/plots",
)
for p in plot_paths:
    print(p)
    display(Image(str(p)))


In [ ]:
# === Final report (markdown) ============================================
from pathlib import Path
from src.analyze import write_report_md
from IPython.display import Markdown, display

report_path = write_report_md(
    comparison_csv="results/comparison_table.csv",
    master_json="results/benchmark_results.json",
    out_md="results/report.md",
)
display(Markdown(Path(report_path).read_text(encoding="utf-8")))


In [ ]:
# === Bundle results =====================================================
import zipfile
from pathlib import Path
from src.analyze import bundle_results

zip_path = bundle_results("results", "banglish_stt_benchmark_results.zip")
size_mb = zip_path.stat().st_size / (1 << 20)
print(f"Bundle: {zip_path}  ({size_mb:.2f} MB)")
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        print(" ", name)

from IPython.display import FileLink, display
display(FileLink(str(zip_path)))
